In [0]:
train = spark.read.table("workspace.default.lc_train_woe")
test  = spark.read.table("workspace.default.lc_test_woe")

print("Train:", train.count(), "| Test:", test.count())
print("Train cols:", train.columns)

Train: 1119710 | Test: 225639
Train cols: ['is_bad', 'issue_year', 'inc_bin_woe', 'dti_bin_woe', 'grade_num_woe', 'home_ownership_idx_woe', 'purpose_idx_woe', 'verification_status_idx_woe', 'issue_year_woe', 'term_months_woe']


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

features_full = [
    "inc_bin_woe", "dti_bin_woe", "grade_num_woe", "home_ownership_idx_woe",
    "purpose_idx_woe", "verification_status_idx_woe", "issue_year_woe", "term_months_woe"
]

assembler_full = VectorAssembler(inputCols=features_full, outputCol="features")
train_full = assembler_full.transform(train)
test_full  = assembler_full.transform(test)

lr_full = LogisticRegression(featuresCol="features", labelCol="is_bad", maxIter=50)
model_full = lr_full.fit(train_full)

print("Intercept:", model_full.interceptVector[0] if hasattr(model_full, 'interceptVector') else model_full.intercept)
for name, coef in zip(features_full, model_full.coefficients):
    print(f"  {name}: {coef:.4f}")

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-5250420644645527>, line 14
     11 test_full  = assembler_full.transform(test)
     13 lr_full = LogisticRegression(featuresCol="features", labelCol="is_bad", maxIter=50)
---> 14 model_full = lr_full.fit(train_full)
     16 print("Intercept:", model_full.interceptVector[0] if hasattr(model_full, 'interceptVector') else model_full.intercept)
     17 for name, coef in zip(features_full, model_full.coefficients):

File /databricks/python_shell/lib/dbruntime/MLWorkloadsInstrumentation/_pyspark.py:30, in _create_patch_function.<locals>.patched_method(self, *args, **kwargs)
     28 call_succeeded = False
     29 try:
---> 30     result = original_method(self, *args, **kwargs)
     31     call_succeeded = True
     32     return result

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/base.py:203, in Estimator.fit(

In [0]:
from pyspark.sql.functions import col, sum as _sum, when

woe_cols = [
    "inc_bin_woe", "dti_bin_woe", "grade_num_woe", "home_ownership_idx_woe",
    "purpose_idx_woe", "verification_status_idx_woe", "issue_year_woe", "term_months_woe"
]

print("=== TRAIN nulls ===")
train.select([_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in woe_cols]).show(truncate=False)

print("=== TEST nulls ===")
test.select([_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in woe_cols]).show(truncate=False)

=== TRAIN nulls ===
+-----------+-----------+-------------+----------------------+---------------+---------------------------+--------------+---------------+
|inc_bin_woe|dti_bin_woe|grade_num_woe|home_ownership_idx_woe|purpose_idx_woe|verification_status_idx_woe|issue_year_woe|term_months_woe|
+-----------+-----------+-------------+----------------------+---------------+---------------------------+--------------+---------------+
|0          |266        |0            |0                     |0              |0                          |0             |0              |
+-----------+-----------+-------------+----------------------+---------------+---------------------------+--------------+---------------+

=== TEST nulls ===
+-----------+-----------+-------------+----------------------+---------------+---------------------------+--------------+---------------+
|inc_bin_woe|dti_bin_woe|grade_num_woe|home_ownership_idx_woe|purpose_idx_woe|verification_status_idx_woe|issue_year_woe|term_months

In [0]:
# Fix 1: drop issue_year from feature list
features_full = [
    "inc_bin_woe", "dti_bin_woe", "grade_num_woe", "home_ownership_idx_woe",
    "purpose_idx_woe", "verification_status_idx_woe", "term_months_woe"
]  # 7 features now, no issue_year

# Fix 2: fill the small dti_bin null pocket with 0 (neutral WoE)
train = train.na.fill(0.0, subset=["dti_bin_woe"])
test  = test.na.fill(0.0, subset=["dti_bin_woe"])

# Confirm clean
from pyspark.sql.functions import col, sum as _sum, when
print("Train nulls:", train.select([_sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in features_full]).collect()[0].asDict())
print("Test nulls:",  test.select([_sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in features_full]).collect()[0].asDict())

Train nulls: {'inc_bin_woe': 0, 'dti_bin_woe': 0, 'grade_num_woe': 0, 'home_ownership_idx_woe': 0, 'purpose_idx_woe': 0, 'verification_status_idx_woe': 0, 'term_months_woe': 0}
Test nulls: {'inc_bin_woe': 0, 'dti_bin_woe': 0, 'grade_num_woe': 0, 'home_ownership_idx_woe': 0, 'purpose_idx_woe': 0, 'verification_status_idx_woe': 0, 'term_months_woe': 0}


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

features_full = [
    "inc_bin_woe", "dti_bin_woe", "grade_num_woe", "home_ownership_idx_woe",
    "purpose_idx_woe", "verification_status_idx_woe", "term_months_woe"
]

assembler_full = VectorAssembler(inputCols=features_full, outputCol="features")
train_full = assembler_full.transform(train)
test_full  = assembler_full.transform(test)

lr_full = LogisticRegression(featuresCol="features", labelCol="is_bad", maxIter=50)
model_full = lr_full.fit(train_full)

print("Intercept:", float(model_full.intercept))
for name, coef in zip(features_full, model_full.coefficients):
    print(f"  {name}: {coef:.4f}")

Intercept: -1.4053920040810939
  inc_bin_woe: -0.5607
  dti_bin_woe: -0.5642
  grade_num_woe: -0.7588
  home_ownership_idx_woe: -0.8901
  purpose_idx_woe: -0.1925
  verification_status_idx_woe: -0.3000
  term_months_woe: -0.5515


In [0]:
train.filter("dti_bin_woe IS NULL").count(), test.filter("dti_bin_woe IS NULL").count()

(0, 0)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import numpy as np

preds_full = model_full.transform(test_full)

# AUC
auc_full = BinaryClassificationEvaluator(labelCol="is_bad", metricName="areaUnderROC").evaluate(preds_full)

# KS statistic — pull probability of default and label to driver
pdf = preds_full.select("is_bad", "probability").toPandas()
pdf["pd"] = pdf["probability"].apply(lambda v: float(v[1]))

pdf_sorted = pdf.sort_values("pd")
pdf_sorted["cum_bad"]  = (pdf_sorted["is_bad"] == 1).cumsum() / (pdf_sorted["is_bad"] == 1).sum()
pdf_sorted["cum_good"] = (pdf_sorted["is_bad"] == 0).cumsum() / (pdf_sorted["is_bad"] == 0).sum()
ks_full = (pdf_sorted["cum_good"] - pdf_sorted["cum_bad"]).abs().max()

gini_full = 2 * auc_full - 1

print(f"FULL MODEL — AUC: {auc_full:.4f} | KS: {ks_full:.4f} | Gini: {gini_full:.4f}")

FULL MODEL — AUC: 0.6923 | KS: 0.2804 | Gini: 0.3847


In [0]:
features_nograde = [f for f in features_full if f != "grade_num_woe"]

assembler_ng = VectorAssembler(inputCols=features_nograde, outputCol="features")
train_ng = assembler_ng.transform(train)
test_ng  = assembler_ng.transform(test)

lr_ng = LogisticRegression(featuresCol="features", labelCol="is_bad", maxIter=50)
model_ng = lr_ng.fit(train_ng)

preds_ng = model_ng.transform(test_ng)
auc_ng = BinaryClassificationEvaluator(labelCol="is_bad", metricName="areaUnderROC").evaluate(preds_ng)

pdf2 = preds_ng.select("is_bad", "probability").toPandas()
pdf2["pd"] = pdf2["probability"].apply(lambda v: float(v[1]))
pdf2_s = pdf2.sort_values("pd")
pdf2_s["cum_bad"]  = (pdf2_s["is_bad"] == 1).cumsum() / (pdf2_s["is_bad"] == 1).sum()
pdf2_s["cum_good"] = (pdf2_s["is_bad"] == 0).cumsum() / (pdf2_s["is_bad"] == 0).sum()
ks_ng = (pdf2_s["cum_good"] - pdf2_s["cum_bad"]).abs().max()
gini_ng = 2 * auc_ng - 1

print(f"NO-GRADE MODEL — AUC: {auc_ng:.4f} | KS: {ks_ng:.4f} | Gini: {gini_ng:.4f}")
print(f"\nDelta from full → no-grade: ΔAUC = {auc_ng - auc_full:+.4f}")

NO-GRADE MODEL — AUC: 0.6453 | KS: 0.2113 | Gini: 0.2906

Delta from full → no-grade: ΔAUC = -0.0470


In [0]:
import math

PDO = 20
BASE = 600
TARGET_ODDS = 50

factor = PDO / math.log(2)
offset = BASE - factor * math.log(TARGET_ODDS)
n_features = len(features_full)
intercept = float(model_full.intercept)
coefs = {name: float(c) for name, c in zip(features_full, model_full.coefficients)}

print(f"factor = {factor:.4f} | offset = {offset:.4f}\n")

# Per-feature points contribution: -(coef * WoE + intercept/n) * factor + offset/n
# Standard scorecard decomposition — each feature contributes a slice of the total score
score_rows = []
for name, coef in coefs.items():
    score_rows.append({
        "feature": name,
        "coefficient": round(coef, 4),
        "points_per_woe_unit": round(-coef * factor, 2)
    })

import pandas as pd
scorecard_df = pd.DataFrame(score_rows).sort_values("points_per_woe_unit", ascending=False)
print(scorecard_df.to_string(index=False))

factor = 28.8539 | offset = 487.1229

                    feature  coefficient  points_per_woe_unit
     home_ownership_idx_woe      -0.8901                25.68
              grade_num_woe      -0.7588                21.89
                dti_bin_woe      -0.5642                16.28
                inc_bin_woe      -0.5607                16.18
            term_months_woe      -0.5515                15.91
verification_status_idx_woe      -0.3000                 8.66
            purpose_idx_woe      -0.1925                 5.56


In [0]:
# Save scored test set for Day 7 (PD calibration → expected loss)
preds_full.select("is_bad", "probability", *features_full) \
    .write.mode("overwrite").saveAsTable("workspace.default.lc_test_scored")

# Save scorecard points table
spark.createDataFrame(scorecard_df) \
    .write.mode("overwrite").saveAsTable("workspace.default.lc_scorecard_points")

print("Saved: lc_test_scored, lc_scorecard_points")

Saved: lc_test_scored, lc_scorecard_points


In [0]:
print(f"FULL:     AUC {auc_full:.4f} | KS {ks_full:.4f} | Gini {gini_full:.4f}")
print(f"NO GRADE: AUC {auc_ng:.4f}   | KS {ks_ng:.4f}   | Gini {gini_ng:.4f}")
print(scorecard_df.to_string(index=False))

FULL:     AUC 0.6923 | KS 0.2804 | Gini 0.3847
NO GRADE: AUC 0.6453   | KS 0.2113   | Gini 0.2906
                    feature  coefficient  points_per_woe_unit
     home_ownership_idx_woe      -0.8901                25.68
              grade_num_woe      -0.7588                21.89
                dti_bin_woe      -0.5642                16.28
                inc_bin_woe      -0.5607                16.18
            term_months_woe      -0.5515                15.91
verification_status_idx_woe      -0.3000                 8.66
            purpose_idx_woe      -0.1925                 5.56
